In [14]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon

In [17]:
# --- CONFIGURATION ---

# --- File Selection ---
EXAMPLES_ENABLED = False    # 3-shot examples
REASONING_ENABLED = False   # Chain-of-thought reasoning
CONFIDENCE_ENABLED = False  # Confidence score
ROLE_NR = 1                 # System prompt choice: 1, 2, 3, or 4
MODEL = "GPT5"

suffix = f"_{MODEL}_role{ROLE_NR}{'_examples' if EXAMPLES_ENABLED else ''}{'_reasoning' if REASONING_ENABLED else ''}{'_confidence' if CONFIDENCE_ENABLED else ''}"
INPUT_FILE = f"regression input{suffix}.csv"

# Choose target variable
TARGET = {1: 'ESG_washing_score', 2: 'relative_ESG_washing_score', 3: 'ESG Controversies Score', 4: 'ESG Score'}[2]

# Define the hierarchical structure of variables
FEATURE_STRUCTURE = {
    'controls': {
        'financials': ['Company Market Cap', 'Return on Equity - Actual', 'Total Debt Percentage of Total Equity'],
        'governance': ['Independent Board Members'],
        # 'roa': ['Return on Assets - Actual']
    },
    'esg_classified': {
        'esg_sentiment': ['esg_sentiment_val'],
        'esg_readability': {
            'esg_subcriteria': [
                'esg_A_01_score', 'esg_A_02_score', 'esg_B_01_score', 'esg_B_02_score', 
                'esg_B_03_score', 'esg_C_01_score', 'esg_C_02_score', 'esg_C_04_score', 
                'esg_C_05_score', 'esg_C_06_score', 'esg_D_01_score'
            ],
            'esg_averages': ['esg_Average_Main_Crit', 'esg_Average_Subcrit'],
            'esg_weighted_averages': ['esg_Weighted_Main_Crit', 'esg_Weighted_Subcrit']
        }
    },
    'total': {
        'tot_sentiment': ['tot_sentiment_val'],
        'tot_readability': {
            'tot_subcriteria': [
                'tot_A_01_score', 'tot_A_02_score', 'tot_B_01_score', 'tot_B_02_score', 
                'tot_B_03_score', 'tot_C_01_score', 'tot_C_02_score', 'tot_C_04_score', 
                'tot_C_05_score', 'tot_C_06_score', 'tot_D_01_score'
            ],
            'tot_averages': ['tot_Average_Main_Crit', 'tot_Average_Subcrit'],
            'tot_weighted_averages': ['tot_Weighted_Main_Crit', 'tot_Weighted_Subcrit']
        }
    }
}

# --- ACTIVE SELECTION ---
# You can add Groups, Subgroups, or Sub-subgroups to this list.
# Examples: 
# - Use 'controls' to enable all financials and governance.
# - Use 'averages' to enable averages for BOTH esg_classified and total.
# - Use 'esg_classified' to enable everything under the ESG prefix.
ENABLED_FEATURES = ['controls', 'esg_Average_Main_Crit', 'tot_Average_Main_Crit']

# Define derived terms for regression
INTERACTION_PAIRS = [] # Insert tuples: (Var1, Var2)
QUADRATIC_FEATURES = [] # Insert variable name

# Choose variables for Wilcoxon test
VAR_1 = 'relative_ESG_washing_score'
VAR_2 = 'esg_Average_Main_Crit'

In [15]:
# --- 2. LOGIC & PROCESSING ---

def get_columns_from_selection(structure, selection):
    """
    Recursively finds column names based on the ENABLED_FEATURES selection.
    """
    cols = []
    for key, value in structure.items():
        # If the key itself is selected, take everything under it
        if key in selection:
            cols.extend(flatten_structure(value))
        # If the value is another dictionary, dive deeper
        elif isinstance(value, dict):
            cols.extend(get_columns_from_selection(value, selection))
    return list(set(cols)) # Remove duplicates

def flatten_structure(obj):
    """Helper to get all strings out of a nested dict/list."""
    items = []
    if isinstance(obj, dict):
        for v in obj.values():
            items.extend(flatten_structure(v))
    elif isinstance(obj, list):
        items.extend(obj)
    return items

def clean_data(df):
    """Handles percentage strings and basic cleaning."""
    df_clean = df.copy()
    # Convert '46.67%' style strings to 0.4667
    for col in df_clean.columns:
        if df_clean[col].dtype == 'object':
            try:
                df_clean[col] = df_clean[col].str.replace('%', '').astype(float) / 100
            except:
                pass
    return df_clean

# Descriptive statistics

In [ ]:
def run_descriptive_analysis(file_path):
    # 1. Data laden
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        return "Bestand niet gevonden. Controleer de bestandsnaam."
    
    df = clean_data(df)
    
    # 2. Kolommen selecteren
    selected_cols = get_columns_from_selection(FEATURE_STRUCTURE, ENABLED_FEATURES)
    all_analysis_cols = [TARGET] + [c for c in selected_cols if c in df.columns]
    
    stats_df = df[all_analysis_cols].dropna()
    
    if stats_df.empty:
        return "Geen data beschikbaar na het verwijderen van lege waarden (NaNs)."

    # 3. Beschrijvende statistiek tonen
    print("--- BESCHRIJVENDE STATISTIEK ---")
    print(stats_df.describe().round(4))
    print("\n")

    # 4. Visualisaties
    sns.set_theme(style="whitegrid")
    
    # Scatterplots: Relatie tussen onafhankelijke variabelen en de Target
    features_only = [c for c in all_analysis_cols if c != TARGET]
    num_features = len(features_only)
    
    if num_features > 0:
        fig, axes = plt.subplots(1, num_features, figsize=(5 * num_features, 5))
        if num_features == 1: axes = [axes] # Zorg dat axes altijd een lijst is
        
        for i, col in enumerate(features_only):
            sns.scatterplot(data=stats_df, x=col, y=TARGET, ax=axes[i])
            axes[i].set_title(f'{col} vs {TARGET}')
        
        plt.tight_layout()
        plt.show()

    # 5. Correlatiematrix (Heatmap)
    plt.figure(figsize=(10, 8))
    sns.heatmap(stats_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Correlatiematrix")
    plt.show()

run_descriptive_analysis(INPUT_FILE)

# Regression

In [ ]:
def run_analysis(file_path):
    # Load data
    df = pd.read_csv(file_path)
    df = clean_data(df)

    # Identify variables to include
    base_features = get_columns_from_selection(FEATURE_STRUCTURE, ENABLED_FEATURES)
    
    # Filter data and drop NaNs for selected features
    model_data = df[[TARGET] + base_features].dropna()
    
    if len(model_data) < 3:
        return f"Error: Too few data points ({len(model_data)}) left after dropping NaNs."

    X = model_data[base_features].copy()
    y = model_data[TARGET]

    # Add Quadratic terms
    for col in QUADRATIC_FEATURES:
        if col in X.columns:
            X[f'{col}_sq'] = X[col] ** 2

    # Add Interaction terms
    for v1, v2 in INTERACTION_PAIRS:
        if v1 in X.columns and v2 in X.columns:
            X[f'{v1}_x_{v2}'] = X[v1] * X[v2]

    # Fit OLS model
    X = sm.add_constant(X) # Add intercept
    model = sm.OLS(y, X.astype(float)).fit()
    
    return model

# --- 3. EXECUTION ---
results = run_analysis(INPUT_FILE)

if isinstance(results, str):
    print(results)
else:
    print(results.summary())

# Wilcoxon

In [19]:
def run_wilcoxon_test(file_path, v1, v2):
    # 1. Data laden
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Fout: Het bestand '{file_path}' is niet gevonden.")
        return

    df = clean_data(df)

    # 2. Controleren of kolommen bestaan
    if v1 not in df.columns or v2 not in df.columns:
        print(f"Fout: Een van de variabelen ({v1} of {v2}) staat niet in het bestand.")
        print(f"Beschikbare kolommen: {list(df.columns)}")
        return

    # 3. Paren maken en NaNs verwijderen
    # De Wilcoxon test vereist dat we voor elke rij beide waarden hebben
    test_data = df[[v1, v2]].dropna()
    
    if len(test_data) < 4:
        print(f"Fout: Te weinig data-paren ({len(test_data)}) voor een betrouwbare test.")
        return

    # 4. De test uitvoeren
    # We vergelijken de twee kolommen
    stat, p_value = wilcoxon(test_data[v1], test_data[v2])

    # 5. Resultaten printen
    print("--- WILCOXON SIGNED-RANK TEST RESULTATEN ---")
    print(f"Variabele 1: {v1} (Gemiddelde: {test_data[v1].mean():.4f})")
    print(f"Variabele 2: {v2} (Gemiddelde: {test_data[v2].mean():.4f})")
    print(f"Aantal paren (n): {len(test_data)}")
    print("-" * 40)
    print(f"Statistiek: {stat:.4f}")
    print(f"p-waarde:   {p_value:.4f}")
    print("-" * 40)

    if p_value < 0.05:
        print("Conclusie: Er is een significant verschil tussen de twee variabelen (p < 0.05).")
    else:
        print("Conclusie: Er is GEEN significant verschil gevonden (p >= 0.05).")

# --- UITVOERING ---
run_wilcoxon_test(INPUT_FILE, VAR_1, VAR_2)

--- WILCOXON SIGNED-RANK TEST RESULTATEN ---
Variabele 1: relative_ESG_washing_score (Gemiddelde: 0.0752)
Variabele 2: esg_Average_Main_Crit (Gemiddelde: 3.3427)
Aantal paren (n): 4
----------------------------------------
Statistiek: 0.0000
p-waarde:   0.1250
----------------------------------------
Conclusie: Er is GEEN significant verschil gevonden (p >= 0.05).
